# 3 · FIR II filtering

**Local Quartus workflow · host or container · one active FPGA design at a time**

`y[n]=Σ h[k]x[n−k]`

A 16-tap FIR performs convolution. Quantized Q15 coefficients, nearest rounding and saturation are part of the hardware contract.

Python model → native VDS diagram → real RTL/IP simulation → SOF → board → focused CyberEther page.

## Setup

Once, from this example folder: `bash scripts/install_local.sh --system` (Linux system packages need sudo), then select **Atum A3 IQ Demos** as the notebook kernel. Quartus/Questa/Nios tools must already be installed. Set the optional paths below if they are not on PATH. No Docker commands are needed.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/lessons.py").exists())
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "cyberether"))
# Optional overrides, set before tool discovery:
# os.environ["QUARTUS_ROOTDIR"] = "/your/install/quartus"
# os.environ["QUESTA_ROOT"] = "/your/install/questa_fse"
# os.environ["ATUM_ALTERA_ROOT"] = "/your/install"
# os.environ["ATUM_UART"] = "/dev/serial/by-id/your-board"
# os.environ["ATUM_JTAG_CABLE"] = "your cable from jtagconfig"
from lessons import Lesson, lan_ip, stop_demo
from common import tool_exe
CHECKPOINT = 3
FORCE_REBUILD = False   # Fresh checkout builds; source-matched SOFs are reused afterwards.
lesson = Lesson(CHECKPOINT)
HOST = lan_ip()
print("Python:", sys.executable)
print("Quartus:", tool_exe("quartus_sh"))
print("Questa:", tool_exe("vsim"))
print("Browser host:", HOST)


## 1 · Python simulation

The code below is the editable numerical model. Its plots are synthetic/theory, separate from the RTL results and live hardware below.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from notebook_plots import spectrum
from nios_fir import PROFILES, fixed_reference
from nios_ddc import reference, gain_reference
from iq_lab_samples import load_sample
raw, _ = load_sample("two-tone")
fs = 16e6
n = CHECKPOINT
coeff = PROFILES[2]
out = fixed_reference(raw, coeff, True)
fig, axs = plt.subplots(1, 3, figsize=(14, 4))
axs[0].stem(np.arange(16), np.array(coeff) / 32768)
axs[0].set(xlabel='Tap', ylabel='Q15 / 32768', title='Actual reloadable Hamming coefficients')
axs[1].plot(raw[:160, 0], label='Before I')
axs[1].plot(out[:160, 0], label='After I')
axs[1].set(xlabel='Sample', ylabel='CI8 count', title='Integer convolution + rounding')
axs[1].legend()
spectrum(axs[2], raw.astype(float), fs, 'Input')
spectrum(axs[2], out.astype(float), fs, 'FIR output')
summary = {'taps': 16, 'reference': 'Python fixed-point convolution; nearest rounding and saturation'}
fig.suptitle(f"{CHECKPOINT} · Python simulation / integer reference")
fig.tight_layout()
plt.show()
summary


## 2 · Generate the actual FPGA block design

Rebuild the Nios initialization memory where present, recreate the native VDS system, validate and generate vendor IP. The PNG/SVG below comes from **Quartus-exported native connections**, rendered by Graphviz; it is not a hand-drawn or simulated topology. Clock/reset fanout is hidden to keep it readable.

In [ ]:
diagram_png = lesson.generate()
print("Exported:", diagram_png)

## 3 · RTL/IP simulation

Actual generated FIR II VHDL plus capture controller; impulse, custom coefficients, rounding and saturation.

The plots below read the actual simulator output files. Assertions must pass before the result is plotted.

In [ ]:
simulation_result = lesson.simulate()
print("Simulation:", simulation_result["status"])
lesson.simulation_plot()

### Open the Questa GUI

Use a desktop terminal **on the machine running Quartus** (or its container desktop). Execute the printed commands. `wave.wlf` opens the saved real simulation; `view.do` adds its signals. To rerun interactively, start `vsim -gui` in the same directory and type `do gui.do` in the Transcript. This reruns the same assertions and leaves the waveform window open.

In [ ]:
lesson.questa_gui()

## 4 · Build the SOF

A source-matched, hash-verified cached bitfile can be reused for a quick presentation. Set `FORCE_REBUILD=True` to demonstrate a full Quartus compilation. A fresh checkout always requires a build.

In [ ]:
build = lesson.build(force=FORCE_REBUILD)
print("SOF:", lesson.stage / "iq_bridge_vds.sof")
print("SHA256:", build["sha256"])
print("Compiled resources:", build["resources"])

## 5 · Program the physical board

This cell **programs the selected checkpoint**. It first stops a previous notebook-owned demo and releases UART/JTAG. Stop any unrelated acquisition service before running it; an unknown serial owner is refused. No ESP32 flashing or reset-button presses are needed with the installed suite.

In [ ]:
lesson.program()

## 6 · Focused CyberEther demo

Toggle Hamming and Boxcar. Each displayed filtered result is independently checked against the configured integer coefficients.

The page uses the bitfile just programmed. One board and one acquisition owner are shared across all notebooks.

In [ ]:
live = lesson.start_demo(host=HOST)
print(live)

Leave the page running while presenting. The next notebook stops it before programming. To stop it manually, run `stop_demo()` in the cell below. Live invites and captures stay in ignored runtime folders; do not publish their tokens.

In [ ]:
# stop_demo()